## Bronze Ingestion – Transactions Incremental (DLT Streaming)

### Purpose
The `transactions` dataset was split into two ingestion patterns to simulate a real production pipeline:

- **Historical batch data (16 months)** was ingested using `COPY INTO`
- **New / incremental transactions** were ingested using a **DLT streaming table**

This approach demonstrates both batch backfill ingestion and incremental ingestion patterns.

---

### DLT Streaming Table
A DLT streaming table was created to ingest incremental transaction CSV files:

- reads files incrementally from a Unity Catalog Volume folder
- processes only new files using DLT-managed checkpoints
- enriches each row with audit and lineage metadata

---

### Bronze Columns
The streaming table stores raw business columns along with standard Bronze audit metadata:

- `loaded_at`
- `updated_at`
- `load_dt`
- `source` (folder-level lineage)
- `source_file` (file-level lineage)

---

### Rerun Safety
The ingestion is rerun-safe because DLT streaming tables track file ingestion state.  
Re-running the pipeline does not reprocess already ingested files, preventing duplicates.


In [0]:
-- DLT streaming table to ingest incremental transaction CSV files

CREATE OR REFRESH STREAMING TABLE coffee.bronze.transactions_incremental
COMMENT 'Incremental transactions ingested via DLT'
AS
SELECT
  -- Raw business columns (kept as STRING to preserve raw data)
  transaction_id,
  store_id,
  payment_method_id,
  voucher_id,
  user_id,
  original_amount,
  discount_applied,
  final_amount,
  created_at,

  -- metadata
  current_timestamp() AS loaded_at,
  current_timestamp() AS updated_at,
  current_date() AS load_dt,
  'transactions_incremental_csv' AS source,
  _metadata.file_path AS source_file

FROM STREAM read_files(
  '/Volumes/workspace/default/coffee_raw_volume/chunked_transactions/incremental/',
  format => 'csv',
  header => true
);
